# URL Uploader Bot — Compact Colab

Set these in **Colab → Secrets** and enable notebook access: `BOT_TOKEN`, `API_ID`, `API_HASH`, `OWNER_ID`, and `DATABASE_URL` (Neon; recommended for persistent history/resume). Optional: `LOG_CHANNEL`, `ADMIN`, `BANNED_USERS`, `HTTP_PROXY`, `CHUNK_SIZE`, `DEF_THUMB_NAIL_VID_S`.

Run the **three cells below in order**. This notebook pulls the latest `main` branch, checks required features, then starts the owner-only bot. Stop the last cell to stop the bot; Colab can disconnect.

**Commands:** `/vid`, `/channel`, `/actress`, `/channels` (saved channels processed one at a time), `/resume`, `/setdump`. Batch speed can be set in Cell 1 with `PARALLEL_SPEED = "safe"`, `"balanced"`, or `"fast"` (1, 2, or 3 concurrent downloads and uploads); the Telegram speed button can also switch modes. Batch mode includes retries, progress, and Neon-backed duplicate skipping/resume. For dump uploads, the bot must be an admin with posting permission in the target channel.

In [ ]:
# 1) Load secrets, install dependencies, and get the latest main branch
import os
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

# Parallel batch profile: safe = 1, balanced = 2, fast = 3 concurrent downloads/uploads.
PARALLEL_SPEED = "balanced"  # Change to "safe" or "fast" as needed.
PARALLEL_SPEED = str(PARALLEL_SPEED).strip().lower()
if PARALLEL_SPEED not in {"safe", "balanced", "fast"}:
    raise ValueError("PARALLEL_SPEED must be safe, balanced, or fast.")
os.environ["PARALLEL_SPEED"] = PARALLEL_SPEED
print(f"Default parallel batch profile: {PARALLEL_SPEED}")



REQUIRED = ("BOT_TOKEN", "API_ID", "API_HASH", "OWNER_ID")
OPTIONAL = ("DATABASE_URL", "LOG_CHANNEL", "ADMIN", "BANNED_USERS", "HTTP_PROXY", "CHUNK_SIZE", "DEF_THUMB_NAIL_VID_S")
missing = []
for key in REQUIRED + OPTIONAL:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value is not None and str(value).strip():
        os.environ[key] = str(value).strip()
    elif key in REQUIRED:
        missing.append(key)
if missing:
    raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing) + ". Add them in Secrets and enable notebook access.")

try:
    int(os.environ["API_ID"])
    owner_id = int(os.environ["OWNER_ID"])
    if owner_id <= 0:
        raise ValueError
except ValueError as exc:
    raise ValueError("API_ID and OWNER_ID must be numeric; OWNER_ID must be positive.") from exc

db_url = os.environ.get("DATABASE_URL", "").strip()
if db_url and not db_url.startswith(("postgres://", "postgresql://")):
    raise ValueError("DATABASE_URL must be a PostgreSQL connection string. Secret value was not displayed.")
print("Secrets loaded; values are not displayed.")
print("Neon persistence enabled." if db_url else "WARNING: DATABASE_URL is not set; persistent history/resume may not work.")

REPO_DIR = Path("/content/URL-uploader")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", "https://github.com/Telebotfaroff/URL-uploader.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyrogram==2.0.106", "tgcrypto", "requests"], check=True)
print("Setup complete: latest GitHub main cloned and dependencies installed.")

In [ ]:
# 2) Compile Python and verify essential features before startup
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "compileall", "-q", "bot.py", "plugins"], check=True)
javdl = Path("plugins/javdl.py").read_text(encoding="utf-8")
database = Path("plugins/database/database.py").read_text(encoding="utf-8")
downloader = Path("plugins/javdl_core/downloader.py").read_text(encoding="utf-8")

checks = {
    "Colab parallel speed config": "PARALLEL_SPEED" in javdl and "PARALLEL_SPEED_PROFILES" in javdl,
    "dump-channel command": "async def set_dump_command(" in javdl and "dump_chat_id" in javdl,
    "persistent settings": "async def get_setting(" in database and "async def set_setting(" in database,
    "parallel batch pipeline": "asyncio.Semaphore(download_limit)" in javdl and "upload_semaphore=upload_semaphore" in javdl,
    "persistent resume": 'filters.command(["resume"]' in javdl and "save_batch_checkpoint" in database,
    "retries, speed controls, rate limits, and one shared batch message": "for attempt in range(1, 4)" in javdl and 'callback_data="javdl|speed"' in javdl and "PROGRESS_MIN_INTERVAL = 5.0" in javdl and "EDIT_MIN_INTERVAL = 1.2" in javdl and "item_status = await message.reply_text" not in javdl,
    "unique download filenames": "time.time_ns()" in downloader,
    "saved /channels command": 'filters.command(["channels"]' in javdl and "def _load_saved_channel_urls_sync" in javdl and "saved_channels_command" in javdl,
    "sequential saved-channel processing": "active_saved_batches" in javdl and "all_pages=True" in javdl,
}
failed = [name for name, passed in checks.items() if not passed]
for name, passed in checks.items():
    print(("✓ " if passed else "✗ ") + name)
if failed:
    raise RuntimeError("Feature verification failed: " + ", ".join(failed) + ". Check that GitHub main has the expected code.")
print("All source checks passed.")

In [ ]:
# 3) Verify Telegram token and start the bot with live logs
import os
import subprocess
import sys
import requests
from datetime import datetime

token = os.environ["BOT_TOKEN"]
owner_id = int(os.environ["OWNER_ID"])
try:
    response = requests.get(f"https://api.telegram.org/bot{token}/getMe", timeout=20)
    response.raise_for_status()
    result = response.json()
except requests.RequestException as exc:
    raise RuntimeError(f"Could not verify BOT_TOKEN with Telegram: {exc}") from exc
if not result.get("ok"):
    raise RuntimeError("Telegram rejected BOT_TOKEN. Check the BOT_TOKEN Colab Secret.")

bot_info = result["result"]
print(f"Token verified: @{bot_info.get('username', 'unknown')}")
print(f"Owner Telegram user ID: {owner_id} (only this owner should use the bot)")
os.makedirs("DOWNLOADS", exist_ok=True)
log_path = "/content/URL-uploader/colab_bot.log"
print(f"Starting bot; live logs are shown below and saved to {log_path}\n" + "-" * 72, flush=True)

process = subprocess.Popen([sys.executable, "-u", "bot.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
try:
    with open(log_path, "a", encoding="utf-8", buffering=1) as log_file:
        log_file.write(f"\n{'=' * 24} Session started {datetime.now().isoformat(timespec='seconds')} {'=' * 24}\n")
        for line in process.stdout:
            log_file.write(line)
            print(line, end="", flush=True)
    print(f"\nBot process exited with code {process.wait()}.", flush=True)
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    print("\nBot stopped.")
finally:
    if process.stdout:
        process.stdout.close()
print("Saved logs: !tail -n 200 /content/URL-uploader/colab_bot.log")